# LA-ef GPU preprocessing (Google Colab, T4)

Produces, per dataset, the small artifacts the serving machine needs (`IMPLEMENTATION_PLAN.md` §4, §7):

| File | Content |
|---|---|
| `gt/neighbors.npy` + `gt/gt_meta.json` | exact top-100 neighbours of the test queries (`gt_source` = provided or generated) |
| `calibration/samplings.bin` + `calib_meta.json` | calibration queries + exact GT (Ada-ef format) |
| `stats/estimator.bin` | mean / covariance (Ada-ef Estimator format) |
| `manifest.json` | fingerprints checked by `scripts/prepare_dataset.sh` |

The large `train.npy` and `test.npy` files stay on Colab's local disk (`DATA_ROOT`); only the small files go to Google Drive (`OUT`).

**Colab limits:** ~16 GB VRAM, ~12.7 GB RAM, ~100 GB local disk. All heavy steps stream in chunks, so the per-dataset requirement is disk space for the raw download plus the `.npy` matrices (dbpedia ≈ 9.5 + 6 GB, deep-image ≈ 3.6 + 3.8 GB). Set `DELETE_LOCAL_AFTER = True` to free disk between datasets.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!nvidia-smi
!df -h /content | tail -1
!free -g | head -2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Get the code

Only `Implementation/gpu_preprocessing/` and `Implementation/configs/datasets.yaml` are needed. Use **one** of the two options:

* **A. git:** set `REPO_URL` (and `REPO_SUBDIR` if `Implementation/` is not the repository root).
* **B. Drive:** upload the `Implementation/` folder (or just those two paths) to Drive and set `DRIVE_CODE_DIR`.

In [ ]:
import pathlib, shutil, subprocess

REPO_URL = ''                 # option A, e.g. 'https://github.com/<user>/<repo>.git'
REPO_SUBDIR = 'Implementation'  # path of Implementation/ inside the repository ('' if it is the root)
DRIVE_CODE_DIR = '/content/drive/MyDrive/ECE-750/Implementation'  # option B

CODE = pathlib.Path('/content/laef')
if CODE.exists():
    shutil.rmtree(CODE)

if REPO_URL:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, '/content/repo'], check=True)
    src = pathlib.Path('/content/repo') / REPO_SUBDIR
else:
    src = pathlib.Path(DRIVE_CODE_DIR)

assert (src / 'gpu_preprocessing' / 'run_preprocess.py').exists(), f'gpu_preprocessing not found under {src}'
assert (src / 'configs' / 'datasets.yaml').exists(), f'configs/datasets.yaml not found under {src}'
shutil.copytree(src / 'gpu_preprocessing', CODE / 'gpu_preprocessing')
(CODE / 'configs').mkdir(parents=True)
shutil.copy(src / 'configs' / 'datasets.yaml', CODE / 'configs' / 'datasets.yaml')
print('code ready in', CODE)

In [ ]:
# Colab ships torch with CUDA; uv keeps the preinstalled build and installs the rest quickly.
!pip install -q uv
!uv pip install --system -q -r /content/laef/gpu_preprocessing/requirements.txt
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')

## 2. Settings

Defaults match `configs/defaults.yaml` (`k`, `calibration.n`, `gt.k_store`). `VERIFY_PROVIDED` re-computes exact GT for that many random test queries of datasets that ship ground truth and fails if agreement < 0.999 (plan P1.6).

In [ ]:
DATASETS = [
    # Provided GT (glove, deep, lastfm) is copied from the raw file by the convert step, so the gt step is skipped for these:
    'glove-100-angular',      # provided GT
    'deep-image-96-angular',  # provided GT
    'dbpedia-openai-1m',      # generated GT
    'synthetic-small',        # generated GT (synthetic, tests and smoke only; made on GPU on Colab)
    'cluster-zipf-100d',      # generated GT (synthetic)
    'coco-i2i-512-angular',   # generated GT (provided file disagrees with exact cosine, Pass 7)
    'coco-t2i-512-angular',   # generated GT (provided file disagrees with exact cosine, Pass 7)
    'lastfm-64-dot',          # provided GT
]
PROVIDED_GT = {'glove-100-angular', 'deep-image-96-angular', 'lastfm-64-dot'}

DATA_ROOT = '/content/data'                          # raw downloads + train.npy / test.npy (local disk)
OUT = '/content/drive/MyDrive/laef_artifacts'        # small artifacts (Drive)
REGISTRY = '/content/laef/configs/datasets.yaml'
DEVICE = 'cuda'
K = 10            # serving k (calibration GT width)
K_STORE = 100     # neighbours stored in gt/neighbors.npy
CALIB_N = 1000    # calibration queries
CALIB_SOURCE = 'query_holdout'  # authors' default; or 'query_holdout'
VERIFY_PROVIDED = 500
STEPS = 'download,convert,gt,calib,stats,manifest'
DELETE_LOCAL_AFTER = False    # free local disk after each dataset (raw files + .npy matrices)

pathlib.Path(OUT).mkdir(parents=True, exist_ok=True)

## 3. Run the pipeline

Every step is idempotent: re-running the cell after a disconnect skips finished steps (pass `--force` to redo one).

In [ ]:
import sys, time

def run_dataset(ds):
    cmd = [sys.executable, 'run_preprocess.py',
           '--dataset', ds, '--steps', STEPS,
           '--data-root', DATA_ROOT, '--out', OUT, '--registry', REGISTRY,
           '--device', DEVICE, '--k', str(K), '--k-store', str(K_STORE),
           '--calib-n', str(CALIB_N), '--calib-source', CALIB_SOURCE]
    if ds in PROVIDED_GT and VERIFY_PROVIDED:
        cmd += ['--verify-provided', str(VERIFY_PROVIDED)]
    print('$', ' '.join(cmd), flush=True)
    t0 = time.time()
    # stream output live; raises CalledProcessError on failure
    with subprocess.Popen(cmd, cwd='/content/laef/gpu_preprocessing', stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
        for line in p.stdout:
            print(line, end='')
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)
    print(f'--- {ds} done in {(time.time() - t0) / 60:.1f} min\n', flush=True)

failed = []
for ds in DATASETS:
    try:
        run_dataset(ds)
    except subprocess.CalledProcessError as e:
        print(f'!!! {ds} failed with exit code {e.returncode}')
        failed.append(ds)
        continue
    if DELETE_LOCAL_AFTER:
        shutil.rmtree(pathlib.Path(DATA_ROOT) / 'raw' / ds, ignore_errors=True)
        shutil.rmtree(pathlib.Path(DATA_ROOT) / 'artifacts' / ds, ignore_errors=True)
        !df -h /content | tail -1

print('failed:', failed or 'none')

## 4. Check the outputs

In [ ]:
import json
import numpy as np

required = ['gt/neighbors.npy', 'gt/gt_meta.json', 'calibration/samplings.bin', 'calibration/calib_meta.json',
            'stats/estimator.bin', 'manifest.json']
for ds in DATASETS:
    d = pathlib.Path(OUT) / ds
    print(f'== {ds}')
    missing = [r for r in required if not (d / r).exists()]
    if missing:
        print('   MISSING:', missing)
        continue
    nb = np.load(d / 'gt' / 'neighbors.npy', mmap_mode='r')
    gt_meta = json.loads((d / 'gt' / 'gt_meta.json').read_text())
    assert nb.ndim == 2 and nb.dtype == np.int32, f'{ds}: gt/neighbors.npy must be an int32 matrix'
    assert {'gt_source', 'K', 'metric'} <= gt_meta.keys(), f'{ds}: gt_meta.json lacks gt_source, K or metric'
    assert nb.shape[1] >= K, f'{ds}: stored GT width {nb.shape[1]} is below the serving k = {K}'
    print('   gt/neighbors.npy', nb.shape, nb.dtype, '| gt_meta', gt_meta)
    meta = json.loads((d / 'calibration' / 'calib_meta.json').read_text())
    print('   calibration:', {k: meta[k] for k in meta if not isinstance(meta[k], list)})
    man = json.loads((d / 'manifest.json').read_text())
    print('   fingerprint', str(man.get('fingerprint'))[:16], '| files', len(man.get('files', {})))
    for r in required:
        print(f'   {r:32s} {(d / r).stat().st_size / 1e6:9.2f} MB')

## 4b. Check provided GT against exact search (optional)

Run this only when a dataset with provided GT fails the `VERIFY_PROVIDED` check. It compares 500 test queries of one dataset three ways: the provided GT, exact search in float64 on the CPU, and the GPU result of `exact_knn`. Run section 2 first, because the cell uses `DATA_ROOT` and `OUT`. Use it for COCO and LastFM. For glove, deep-image and dbpedia the float64 arrays are too large for Colab's RAM. The cell does not change any file.

How to read the printed numbers:

- A float16 share near 1.0 suggests the provided distances were rounded to float16, so the provided GT was computed in float16.
- Provided vs float64 near 0.99, with GPU vs float64 near 1.0, means the provided GT differs from exact search and the GPU result is correct.
- GPU vs float64 near 0.99 means float32 arithmetic on the GPU reorders close neighbours.
- Duplicate base rows above 0 means identical vectors, which can swap at rank 10 without a real error.

In [ ]:
import sys, pathlib
import numpy as np
sys.path.insert(0, '/content/laef/gpu_preprocessing')
from prep.ground_truth import exact_knn
from prep.sources import NpySource

ds = 'coco-i2i-512-angular'   # or 'coco-t2i-512-angular', 'lastfm-64-dot'
local = pathlib.Path(DATA_ROOT) / 'artifacts' / ds
gt = pathlib.Path(OUT) / ds / 'gt'
K = 10

train = np.load(local / 'train.npy')
test = np.load(local / 'test.npy')
prov_ids = np.load(gt / 'neighbors.npy', mmap_mode='r')
prov_d = np.asarray(np.load(gt / 'distances.npy', mmap_mode='r')[:, :K], dtype=np.float32)
rows = np.sort(np.random.default_rng(12345).choice(test.shape[0], size=500, replace=False))  # same rows as verify

def unit(x):
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-30)

B = unit(train.astype(np.float64))
Q = unit(test[rows].astype(np.float64))
cpu64 = np.argsort(1.0 - Q @ B.T, axis=1, kind='stable')[:, :K]   # exact search in float64 on CPU
gpu, _ = exact_knn(test[rows], NpySource(local / 'train.npy'), K, 'cd', device='cuda')
P = np.asarray(prov_ids[rows, :K])

def recall(a, b):
    return np.mean([len(set(x) & set(y)) / K for x, y in zip(a, b)])

print('provided distances that are float16 values:', np.mean(prov_d.astype(np.float16).astype(np.float32) == prov_d))
print('duplicate base rows:', len(train) - len(np.unique(train, axis=0)))
print('provided vs float64 CPU:', recall(P, cpu64))
print('GPU vs float64 CPU:     ', recall(gpu, cpu64))
print('GPU vs provided:        ', recall(gpu, P))

## 5. Copy the outputs to the serving machine (WSL)

Only the small files go to Drive. The large `train.npy` and `test.npy` stay on Colab and are rebuilt on the serving machine.

| Output | Colab location | Destination on WSL |
|---|---|---|
| `gt/neighbors.npy`, `gt/gt_meta.json` | `OUT/<dataset>/gt/` | `data/artifacts/<dataset>/gt/` |
| `calibration/samplings.bin`, `calibration/calib_meta.json` | `OUT/<dataset>/calibration/` | `data/artifacts/<dataset>/calibration/` |
| `stats/estimator.bin` | `OUT/<dataset>/stats/` | `data/artifacts/<dataset>/stats/` |
| `manifest.json` | `OUT/<dataset>/manifest.json` | `data/artifacts/<dataset>/manifest.json` |
| `train.npy`, `test.npy`, `holdout.npy` | `DATA_ROOT/artifacts/<dataset>/` (not copied) | rebuilt by `prepare_dataset.sh` |

`OUT` is `/content/drive/MyDrive/laef_artifacts`, so each dataset is one folder `laef_artifacts/<dataset>/` on Drive. The Colab paths `DATA_ROOT` and `OUT` are set in section 2.

**Steps**

1. Check the section 4 output. Every dataset must show no `MISSING` line. Rerun a dataset that has one.
2. Download the folder `laef_artifacts/<dataset>/` from Drive (Google Drive for desktop, or drive.google.com, then Download) into a Windows folder, for example `C:\Users\rizvi\Downloads\laef_artifacts\<dataset>\`.
3. In the WSL terminal, from the `Implementation/` folder, run:

```bash
bash scripts/prepare_dataset.sh <dataset> /mnt/c/Users/rizvi/Downloads/laef_artifacts/<dataset>
```

This rebuilds `train.npy`, `test.npy` and `holdout.npy` in `data/artifacts/<dataset>/` from the raw file in `data/raw/<dataset>/`, downloading that file if it is missing. It then checks the fingerprints in `manifest.json`. If a check fails, nothing is copied. Otherwise it copies `gt/`, `calibration/`, `stats/` and `manifest.json` into `data/artifacts/<dataset>/`.

4. Run the index, calibration and profile steps:

```bash
bash scripts/run_pipeline.sh <dataset> <threads>
```

**Notes**

- Repeat steps 2 to 4 for each dataset.
- All eight datasets are listed in `DATASETS`, including `synthetic-small`, which the GPU generates like the others. The provided-GT datasets (glove, deep, lastfm) download their raw files in the download step. Their GT is copied in the convert step, so the gt step is skipped. The verification still runs when `VERIFY_PROVIDED` is set. The COCO datasets use GT generated on the GPU, because their provided GT disagrees with exact cosine search (Pass 7 in `VERIFICATION_NOTES.md`). On Drive, delete `laef_artifacts/coco-*/gt/` before rerunning, because the gt step skips a dataset whose `neighbors.npy` already exists.